# E3 — ARPG-XL Generalization (FID-50K, n=3)

**Goal.** Show the RTR headline numbers reproduce on the 719M-parameter ARPG-XL model. The strongest single defense against "this is ARPG-L specific."

**Configs.** 18 FID-50K runs:
- Vanilla ARPG-XL at steps {8, 16, 32} × 3 seeds = 9 runs
- RTR (random, ρ=0.7) ARPG-XL at the same step counts and seeds = 9 runs

**Settings (paper's recommended for ARPG-XL).** CFG=6.0 (vs ARPG-L's 5.0), arccos schedule, linear CFG schedule, bf16, single A100. Reproducing ARPG paper's vanilla numbers requires CFG=6.0 for ARPG-XL — we use the paper's standard rather than re-tuning per model.

**Why ρ=0.7 unchanged.** The cap was tuned on ARPG-L (RTR Phase 1). We apply it unchanged to ARPG-XL because (a) the structural-deferral mechanism doesn't depend on model size, (b) re-running the full cap sweep on ARPG-XL would double compute. Limitation noted for the paper.

**Compute.** ARPG-XL is ~1.6× slower than ARPG-L per forward pass. 18 × ~50 min average ≈ **~15–20 hours**, fits in 2 overnight Colab Pro sessions.

**Persistence.** Same resumability pattern as `main_table_colab.ipynb`. Master CSV at `MyDrive/ARPG-assets/results/final-paper/arpgxl-main-table/results.csv`. NPZs NOT kept on Drive (`KEEP_NPZ_ON_DRIVE = False`).

## 1. Mount Drive and set up paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/ARPG-assets')

# Output location for the §5 ARPG-XL generalization table
RESULTS_ROOT = DRIVE_ROOT / 'results' / 'final-paper' / 'arpgxl-main-table'
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
CSV_PATH = RESULTS_ROOT / 'results.csv'
LOG_DIR = RESULTS_ROOT / 'logs'
LOG_DIR.mkdir(parents=True, exist_ok=True)

# Qualitative samples + side-by-side comparison figures
SAMPLES_DIR = RESULTS_ROOT / 'samples'
SAMPLES_GRIDS_DIR = SAMPLES_DIR / 'grids'
SAMPLES_INDIVIDUAL_DIR = SAMPLES_DIR / 'individual'
SAMPLES_GRIDS_DIR.mkdir(parents=True, exist_ok=True)
SAMPLES_INDIVIDUAL_DIR.mkdir(parents=True, exist_ok=True)

# Rejection tracker output (RTR configs only)
REJECTION_LOGS_DIR = RESULTS_ROOT / 'rejection-logs'
REJECTION_LOGS_DIR.mkdir(parents=True, exist_ok=True)

# NPZ persistence policy — NEVER keep 10 GB NPZs on Drive
KEEP_NPZ_ON_DRIVE = False

# Repo + local sampling scratch space
REPO_LOCAL = Path('/content/ARPG-main')
LOCAL_SAMPLE_DIR = Path('/content/samples')
LOCAL_SAMPLE_DIR.mkdir(parents=True, exist_ok=True)

# Fixed assets
REF_NPZ = DRIVE_ROOT / 'eval' / 'VIRTUAL_imagenet256_labeled.npz'
ARPG_L_CKPT = DRIVE_ROOT / 'weights' / 'arpg_300m.pt'     # legacy; not used here
ARPG_XL_CKPT = DRIVE_ROOT / 'weights' / 'arpg_700m.pt'    # NEW for this notebook
VQ_CKPT = DRIVE_ROOT / 'weights' / 'vq_ds16_c2i.pt'
GD_REPO = DRIVE_ROOT / 'external' / 'guided-diffusion'

print(f'Drive root  : {DRIVE_ROOT}')
print(f'Results root: {RESULTS_ROOT}')
print(f'Master CSV  : {CSV_PATH}')
print(f'ARPG-XL ckpt: {ARPG_XL_CKPT} (exists: {ARPG_XL_CKPT.exists()})')
print(f'KEEP_NPZ_ON_DRIVE = {KEEP_NPZ_ON_DRIVE}')

## 2. Clone repo and verify assets (auto-downloads `arpg_700m.pt` if missing)

In [ ]:
REPO_URL = 'https://github.com/rshahbazov23/comp447-arpg-private.git'
GITHUB_TOKEN = None  # e.g. 'ghp_...'

import subprocess, shutil

def _clone_url(url, token):
    if token and url.startswith('https://github.com/'):
        return url.replace('https://', f'https://{token}@')
    return url

# --- 1. Clone / update the ARPG fork ---------------------------------------
if not REPO_LOCAL.exists():
    print(f'Cloning {REPO_URL} → {REPO_LOCAL}')
    subprocess.run(['git', 'clone', _clone_url(REPO_URL, GITHUB_TOKEN), str(REPO_LOCAL)], check=True)
else:
    print('Repo present, pulling latest')
    subprocess.run(['git', '-C', str(REPO_LOCAL), 'pull'], check=True)

# --- 2. Auto-setup assets (downloads anything missing) --------------------
ASSET_URLS = {
    REF_NPZ:      'https://openaipublic.blob.core.windows.net/diffusion/jul-2021/ref_batches/imagenet/256/VIRTUAL_imagenet256_labeled.npz',
    ARPG_XL_CKPT: 'https://huggingface.co/hp-l33/ARPG/resolve/main/arpg_700m.pt',
    VQ_CKPT:      'https://huggingface.co/FoundationVision/LlamaGen/resolve/main/vq_ds16_c2i.pt',
}

ALT_LOCATIONS = [
    Path('/content/drive/MyDrive/eval'),
    Path('/content/drive/MyDrive/weights'),
    Path('/content/drive/MyDrive/ARPG/eval'),
    Path('/content/drive/MyDrive/ARPG/weights'),
]

def find_alt(filename):
    for root in ALT_LOCATIONS:
        cand = root / filename
        if cand.exists():
            return cand
    return None

for target, url in ASSET_URLS.items():
    if target.exists():
        size_gb = target.stat().st_size / 1e9
        print(f'OK   {target.name:<40} ({size_gb:.2f} GB)')
        continue
    alt = find_alt(target.name)
    if alt:
        print(f'Found {target.name} at {alt} — copying')
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(alt, target)
        continue
    print(f'Downloading {target.name} → {target}')
    target.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(['wget', '--show-progress', '-O', str(target), url], check=True)
    print(f'   downloaded ({target.stat().st_size/1e9:.2f} GB)')

# --- 3. guided-diffusion ---------------------------------------------------
if not GD_REPO.exists():
    GD_REPO.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(['git', 'clone', 'https://github.com/openai/guided-diffusion.git', str(GD_REPO)], check=True)

# --- 4. Sanity check -------------------------------------------------------
for p, name in [
    (REF_NPZ,      'ImageNet reference batch'),
    (ARPG_XL_CKPT, 'ARPG-XL pretrained checkpoint'),
    (VQ_CKPT,      'VQ tokenizer'),
    (GD_REPO / 'evaluations' / 'evaluator.py', 'guided-diffusion evaluator'),
    (REPO_LOCAL / 'sample_c2i_ddp.py', 'ARPG sampling script'),
    (REPO_LOCAL / 'models' / 'arpg.py', 'ARPG model'),
    (REPO_LOCAL / 'models' / 'confidence.py', 'Confidence module'),
]:
    if not p.exists():
        raise FileNotFoundError(f'MISSING: {name} → {p}')

conf_src = (REPO_LOCAL / 'models' / 'confidence.py').read_text()
if 'random_score' not in conf_src:
    raise RuntimeError('Random support missing — push 605038a+ from local Mac.')

print('\nAll assets present, random support confirmed.')

## 3. Install dependencies

In [ ]:
subprocess.run(['pip', 'install', '-q',
    'einops', 'transformers', 'scipy', 'tensorflow', 'pandas',
], check=True)

import torch, pandas as pd
print(f'torch: {torch.__version__}, GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE"}')
assert torch.cuda.is_available(), 'No CUDA — switch the Colab runtime to GPU.'

## 4. Config matrix and master CSV

**Priority order**: 16 steps first (the headline regime — anchors the §5 generalization claim), then 8 steps (extreme regime), then 32 steps (regime-boundary control).

In [ ]:
import pandas as pd
from datetime import datetime

MODEL_NAME = 'ARPG-XL'
MODEL_CKPT_NAME = 'arpg_700m'
CFG_SCALE = 6.0   # paper's recommended for ARPG-XL

STEP_COUNTS_ORDER = [16, 8, 32]
SEEDS = [0, 1, 2]

# RTR hyperparameters — applied unchanged from ARPG-L (RTR Phase 1 optimum).
# Limitation noted in the paper: cap was tuned on ARPG-L, applied as-is to ARPG-XL.
RTR_CAP = 0.7
RTR_THRESHOLD = 2.0
RTR_METRIC = 'random'

# Canonical ImageNet classes for qualitative figures (matches ARPG paper's quick-start)
QUALITATIVE_CLASSES = [207, 360, 388, 113, 355, 980, 323, 979]

def make_configs():
    out = []
    for step in STEP_COUNTS_ORDER:
        for mode in ['vanilla', 'rtr']:
            for seed in SEEDS:
                out.append({'mode': mode, 'step': step, 'seed': seed})
    return out

CONFIGS = make_configs()
print(f'Config matrix: {len(CONFIGS)} configs (3 steps × 2 modes × 3 seeds)')

# Initialise master CSV (no prior results — ARPG-XL is fresh)
if not CSV_PATH.exists():
    pd.DataFrame(columns=[
        'mode', 'step', 'seed', 'fid',
        'inception_score', 'sfid', 'precision', 'recall',
        'npz_path', 'source', 'timestamp',
    ]).to_csv(CSV_PATH, index=False)
    print(f'Initialised empty master CSV: {CSV_PATH}')
else:
    df_master = pd.read_csv(CSV_PATH)
    print(f'Master CSV has {len(df_master)} existing rows.')

df_master = pd.read_csv(CSV_PATH)
done_keys = set((r['mode'], int(r['step']), int(r['seed']))
                for _, r in df_master.iterrows()) if len(df_master) else set()
remaining = [c for c in CONFIGS
             if (c['mode'], c['step'], c['seed']) not in done_keys]
print(f'\nProgress: {len(CONFIGS) - len(remaining)}/{len(CONFIGS)} done, {len(remaining)} remaining')
if remaining[:6]:
    print('Next 6 configs:')
    for c in remaining[:6]:
        print(f'  mode={c["mode"]:<7} step={c["step"]:>2} seed={c["seed"]}')

## 5. Helper functions

In [ ]:
import re, time, traceback

def config_to_folder_name(cfg):
    """Mirrors sample_c2i_ddp.py:114 naming convention for ARPG-XL at CFG=6.0."""
    base = (
        f'{MODEL_NAME}-{MODEL_CKPT_NAME}-size-256-size-256-VQ-16-'
        f'topk-0-topp-1.0-temperature-1.0-cfg-{CFG_SCALE}-cfg-schedule-linear-'
        f'sample-schedule-arccos-step-{cfg["step"]}-seed-{cfg["seed"]}'
    )
    if cfg['mode'] == 'rtr':
        base += f'-mode-rejection-metric-{RTR_METRIC}-tau-{RTR_THRESHOLD}-cap-{RTR_CAP}'
    return base


def is_done(cfg, df_master):
    if not len(df_master):
        return False
    mask = (
        (df_master['mode'] == cfg['mode'])
        & (df_master['step'].astype(int) == cfg['step'])
        & (df_master['seed'].astype(int) == cfg['seed'])
    )
    return bool(mask.any())


def cleanup_local_samples():
    if LOCAL_SAMPLE_DIR.exists():
        shutil.rmtree(LOCAL_SAMPLE_DIR)
    LOCAL_SAMPLE_DIR.mkdir(parents=True, exist_ok=True)


def run_sampling(cfg, log_handle=None):
    folder = config_to_folder_name(cfg)
    cmd = [
        'torchrun', '--nnodes=1', '--nproc_per_node=1',
        'sample_c2i_ddp.py',
        '--gpt-model', MODEL_NAME,
        '--gpt-ckpt', str(ARPG_XL_CKPT),
        '--vq-ckpt', str(VQ_CKPT),
        '--sample-schedule', 'arccos',
        '--cfg-schedule', 'linear',
        '--cfg-scale', str(CFG_SCALE),
        '--step', str(cfg['step']),
        '--per-proc-batch-size', '64',
        '--num-fid-samples', '50000',
        '--global-seed', str(cfg['seed']),
        '--sample-dir', str(LOCAL_SAMPLE_DIR),
        '--no-compile',
        '--precision', 'bf16',
    ]
    if cfg['mode'] == 'rtr':
        folder_for_log = config_to_folder_name(cfg)
        cmd += [
            '--rejection-mode', 'rejection',
            '--confidence-metric', RTR_METRIC,
            '--rejection-threshold', str(RTR_THRESHOLD),
            '--max-reject-rate', str(RTR_CAP),
            '--log-json', str(REJECTION_LOGS_DIR / f'{folder_for_log}.json'),
        ]
    print(f'  Sampling: {MODEL_NAME} mode={cfg["mode"]} step={cfg["step"]} seed={cfg["seed"]} cfg={CFG_SCALE}')
    t0 = time.time()
    proc = subprocess.Popen(cmd, cwd=str(REPO_LOCAL),
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    last_print = t0
    for line in proc.stdout:
        if log_handle is not None:
            log_handle.write(line); log_handle.flush()
        now = time.time()
        if now - last_print > 60:
            print(f'    [{(now-t0)/60:.1f} min] {line.rstrip()[:120]}')
            last_print = now
    proc.wait()
    print(f'  Sampling done in {(time.time()-t0)/60:.1f} min (exit {proc.returncode})')
    if proc.returncode != 0:
        raise RuntimeError(f'Sampling failed for {cfg}: exit {proc.returncode}')
    local_npz = LOCAL_SAMPLE_DIR / f'{folder}.npz'
    if not local_npz.exists():
        raise FileNotFoundError(f'NPZ not produced: {local_npz}')
    return local_npz


_METRIC_LINE = re.compile(r'^\s*(FID|sFID|Inception Score|Precision|Recall)\s*:\s*([0-9.eE+\-]+)')

def evaluate_fid(local_npz, log_handle=None):
    cmd = ['python', 'evaluations/evaluator.py', str(REF_NPZ), str(local_npz)]
    t0 = time.time()
    proc = subprocess.run(cmd, cwd=str(GD_REPO), capture_output=True, text=True)
    print(f'  FID eval done in {(time.time()-t0)/60:.1f} min (exit {proc.returncode})')
    if log_handle is not None:
        log_handle.write('--- evaluator stdout ---\n')
        log_handle.write(proc.stdout)
        log_handle.write('\n--- evaluator stderr ---\n')
        log_handle.write(proc.stderr)
        log_handle.flush()
    if proc.returncode != 0:
        print('STDOUT (tail):', proc.stdout[-2000:])
        print('STDERR (tail):', proc.stderr[-2000:])
        raise RuntimeError(f'FID eval failed: exit {proc.returncode}')
    metrics = {}
    for line in proc.stdout.splitlines():
        m = _METRIC_LINE.match(line)
        if m:
            key = m.group(1).lower().replace(' ', '_')
            metrics[key] = float(m.group(2))
    if 'fid' not in metrics:
        print('STDOUT:', proc.stdout)
        raise ValueError('Could not parse FID')
    return metrics


def save_qualitative_samples(local_npz, cfg, num_classes=1000):
    """Extract one sample per QUALITATIVE_CLASSES from the NPZ, save individual PNGs + 2x4 grid."""
    import numpy as np
    from PIL import Image
    npz = np.load(str(local_npz))
    arr = npz['arr_0']
    folder = config_to_folder_name(cfg)
    indiv_dir = SAMPLES_INDIVIDUAL_DIR / folder
    indiv_dir.mkdir(parents=True, exist_ok=True)
    pils = []
    for cls in QUALITATIVE_CLASSES:
        idx = cls
        if idx >= arr.shape[0]:
            continue
        pil = Image.fromarray(arr[idx])
        pil.save(indiv_dir / f'class{cls:03d}.png')
        pils.append((cls, pil))
    if len(pils) == len(QUALITATIVE_CLASSES):
        w, h = pils[0][1].size
        ncols, nrows = 4, 2
        grid = Image.new('RGB', (w * ncols, h * nrows), color=(255, 255, 255))
        for i, (_, pil) in enumerate(pils):
            r, c = divmod(i, ncols)
            grid.paste(pil, (c * w, r * h))
        grid.save(SAMPLES_GRIDS_DIR / f'{folder}_grid.png')
        print(f'  Saved 8 class samples + grid')


def append_result(cfg, metrics, npz_path):
    df = pd.read_csv(CSV_PATH)
    row = {
        'mode': cfg['mode'],
        'step': cfg['step'],
        'seed': cfg['seed'],
        'fid': metrics.get('fid'),
        'inception_score': metrics.get('inception_score'),
        'sfid': metrics.get('sfid'),
        'precision': metrics.get('precision'),
        'recall': metrics.get('recall'),
        'npz_path': str(npz_path) if npz_path is not None else '(not kept)',
        'source': 'arpgxl-run',
        'timestamp': datetime.now().isoformat(),
    }
    df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
    df.to_csv(CSV_PATH, index=False)
    return df


print('Helpers loaded.')

## 6. Main loop — resumable

Safe to interrupt at any time. Each completed config writes a row to the Drive CSV immediately. Re-running this cell picks up where it left off.

**Heads up.** ARPG-XL is ~1.6× slower than ARPG-L per forward pass. Expect ~50 min per FID-50K config on average. Plan for two overnight Colab Pro sessions.

In [ ]:
failures = []
started = datetime.now()

for i, cfg in enumerate(CONFIGS, 1):
    print(f'\n{"="*70}\n[{i}/{len(CONFIGS)}] config={cfg}\n{"="*70}')

    df_master = pd.read_csv(CSV_PATH)
    if is_done(cfg, df_master):
        print('  SKIP (already in master CSV)')
        continue

    folder = config_to_folder_name(cfg)
    local_npz = LOCAL_SAMPLE_DIR / f'{folder}.npz'
    log_path = LOG_DIR / f'{folder}.log'

    try:
        with open(log_path, 'w') as log_h:
            local_npz = run_sampling(cfg, log_handle=log_h)
            drive_npz = None  # NPZ stays on local disk only (KEEP_NPZ_ON_DRIVE=False)
            print(f'  NPZ kept on local disk only ({local_npz.stat().st_size/1e9:.1f} GB) — will be wiped after eval')

            metrics = evaluate_fid(local_npz, log_handle=log_h)
            append_result(cfg, metrics, drive_npz)

            try:
                save_qualitative_samples(local_npz, cfg)
            except Exception as e:
                print(f'  Qualitative samples skipped: {e}')

        print(f'  DONE   FID={metrics["fid"]:.4f}   IS={metrics.get("inception_score", 0):.2f}   Prec={metrics.get("precision", 0):.3f}   Rec={metrics.get("recall", 0):.3f}')
    except Exception as e:
        print(f'  FAILED: {e}')
        traceback.print_exc()
        failures.append({'config': cfg, 'error': str(e), 'log': str(log_path)})
    finally:
        cleanup_local_samples()

    elapsed_h = (datetime.now() - started).total_seconds() / 3600
    print(f'  Session elapsed: {elapsed_h:.2f} h')

print(f'\n\n{"="*70}\nSESSION COMPLETE\n{"="*70}')
print(f'Configs attempted: {len(CONFIGS)}')
print(f'Failures: {len(failures)}')
for f in failures:
    print(f'  {f["config"]}  →  {f["error"]}  (log: {f["log"]})')

## 7. Summary — ARPG-XL headline table

Mean ± std per (mode, step), deltas vs vanilla, gap-closure metric, and cross-model comparison vs ARPG-L.

In [ ]:
import numpy as np

df = pd.read_csv(CSV_PATH)
if len(df) == 0:
    raise RuntimeError('Master CSV is empty — run the main loop first.')
df['step'] = df['step'].astype(int)
df['seed'] = df['seed'].astype(int)

print(f'Master CSV: {len(df)} rows total\n')

summary = (
    df.groupby(['mode', 'step'])['fid']
      .agg(['mean', 'std', 'count'])
      .round(4)
      .sort_index()
)
print(f'ARPG-XL FID-50K (CFG={CFG_SCALE}, mean ± std, n)\n')
print(summary.to_string())

# Headline deltas (RTR vs vanilla)
print('\n\nARPG-XL headline deltas (RTR vs vanilla, multi-seed mean):')
for step in sorted(df['step'].unique()):
    v = df[(df['mode']=='vanilla') & (df['step']==step)]['fid']
    r = df[(df['mode']=='rtr')     & (df['step']==step)]['fid']
    if len(v) > 0 and len(r) > 0:
        v_mean, r_mean = v.mean(), r.mean()
        delta = r_mean - v_mean
        pct = 100 * delta / v_mean
        print(f'  {step:>3} steps: vanilla={v_mean:.4f} (n={len(v)})  RTR={r_mean:.4f} (n={len(r)})  Δ={delta:+.4f}  ({pct:+.2f}%)')

# Gap-closure (16 → 32 step regime)
v16 = df[(df['mode']=='vanilla') & (df['step']==16)]['fid'].mean()
v32 = df[(df['mode']=='vanilla') & (df['step']==32)]['fid'].mean()
r16 = df[(df['mode']=='rtr')     & (df['step']==16)]['fid'].mean()
if all(not np.isnan(x) for x in [v16, v32, r16]):
    gap = (v16 - r16) / (v16 - v32)
    print(f'\nGap closure: RTR@16 closes {100*gap:.1f}% of the 16→32-step ARPG-XL vanilla quality gap')
    print(f'  vanilla@16 = {v16:.4f}')
    print(f'  RTR@16     = {r16:.4f}')
    print(f'  vanilla@32 = {v32:.4f}')

# Cross-model comparison: ARPG-XL vs ARPG-L (from main-table CSV if available)
ML_MAIN_TABLE_CSV = DRIVE_ROOT / 'results' / 'final-paper' / 'main-table' / 'results.csv'
if ML_MAIN_TABLE_CSV.exists():
    df_l = pd.read_csv(ML_MAIN_TABLE_CSV)
    df_l['step'] = df_l['step'].astype(int)
    print('\n\nCross-model comparison (ARPG-L from main-table vs ARPG-XL from this notebook):')
    print(f'{"":15} | {"ARPG-L mean":>12} | {"ARPG-XL mean":>13} |')
    print(f'{"-"*15} | {"-"*12} | {"-"*13} |')
    for mode in ['vanilla', 'rtr']:
        for step in sorted(df['step'].unique()):
            xl = df[(df['mode']==mode) & (df['step']==step)]['fid'].mean()
            l  = df_l[(df_l['mode']==mode) & (df_l['step']==step)]['fid'].mean()
            label = f'{mode:<7} step={step:>2}'
            xl_s = f'{xl:.4f}' if not np.isnan(xl) else 'n/a'
            l_s = f'{l:.4f}' if not np.isnan(l) else 'n/a'
            print(f'{label:15} | {l_s:>12} | {xl_s:>13} |')
else:
    print(f'\n(ARPG-L main-table CSV not found at {ML_MAIN_TABLE_CSV}; cross-model comparison skipped)')

# Write summary CSV alongside the master
summary_path = RESULTS_ROOT / 'summary.csv'
summary.to_csv(summary_path)
print(f'\nWrote summary table: {summary_path}')

## 8. Compose ARPG-XL vanilla-vs-RTR comparison figures

For each (step, seed) pair where both vanilla and RTR grids exist, stack them side-by-side with FID labels. These go in §5 of the paper as visual evidence that RTR samples are competitive with vanilla on the larger model.

Safe to re-run any time. Only composes for configs that have grids on Drive.

In [ ]:
import numpy as np
from PIL import Image, ImageDraw, ImageFont

COMPARISONS_DIR = SAMPLES_DIR / 'comparisons'
COMPARISONS_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(CSV_PATH)
df['step'] = df['step'].astype(int)
df['seed'] = df['seed'].astype(int)

try:
    FONT = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf', 28)
except Exception:
    FONT = ImageFont.load_default()

def fid_for(mode, step, seed):
    row = df[(df['mode']==mode) & (df['step']==step) & (df['seed']==seed)]
    if len(row) == 0:
        return None
    return float(row.iloc[0]['fid'])

def find_grid(mode, step, seed):
    cfg = {'mode': mode, 'step': step, 'seed': seed}
    name = config_to_folder_name(cfg) + '_grid.png'
    candidate = SAMPLES_GRIDS_DIR / name
    return candidate if candidate.exists() else None

made = 0
skipped = 0
for step in sorted(df['step'].unique()):
    for seed in sorted(df[df['step']==step]['seed'].unique()):
        van_grid_path = find_grid('vanilla', int(step), int(seed))
        rtr_grid_path = find_grid('rtr',     int(step), int(seed))
        if van_grid_path is None or rtr_grid_path is None:
            skipped += 1
            continue

        van_grid = Image.open(van_grid_path)
        rtr_grid = Image.open(rtr_grid_path)
        w, h = van_grid.size
        gap, header = 30, 70
        canvas = Image.new('RGB', (w * 2 + gap, h + header + 20), color=(255, 255, 255))
        canvas.paste(van_grid, (0, header))
        canvas.paste(rtr_grid, (w + gap, header))

        draw = ImageDraw.Draw(canvas)
        van_fid = fid_for('vanilla', int(step), int(seed))
        rtr_fid = fid_for('rtr', int(step), int(seed))
        van_label = f'Vanilla {MODEL_NAME} — {step} steps, seed {seed}'
        rtr_label = f'RTR {MODEL_NAME} — {step} steps, seed {seed}'
        if van_fid is not None: van_label += f'   FID-50K = {van_fid:.3f}'
        if rtr_fid is not None: rtr_label += f'   FID-50K = {rtr_fid:.3f}'
        draw.text((10, 10), van_label, font=FONT, fill='black')
        draw.text((w + gap + 10, 10), rtr_label, font=FONT, fill='black')

        out_path = COMPARISONS_DIR / f'compare_step{step:02d}_seed{seed}.png'
        canvas.save(out_path)
        made += 1

print(f'Composed {made} comparison figures, skipped {skipped} (missing grids).')
print(f'Output: {COMPARISONS_DIR}')

preview = COMPARISONS_DIR / 'compare_step16_seed0.png'
if preview.exists():
    from IPython.display import Image as IPImage, display
    display(IPImage(str(preview)))